# 3D Couette Flow in Couette Rheometer

## Import libraries

In [1]:
import numpy as np
from numba import njit
import numba as nb
import matplotlib.pyplot as plt
from numba import float32,float64, guvectorize 
#from cv2 import threshold
from matplotlib.pylab import f

### Función radio de equilibrio

In [2]:
def compute_equilibrium_radius(G, H, p, q):
    # Solución analítica directa
    r_equilibrio = (H / G) ** (1 / (q - p))
    print(f"El radio de equilibrio exacto es: {r_equilibrio} cm")
    return r_equilibrio

## Parámetros iniciales 

In [3]:
# Fuerza 
G = 0.3  
H = 0.00075 
p = 3
q = 5 
m = 1.0
# Radios
R_in = 1.255     
R_out = 1.355    
r_0 = compute_equilibrium_radius(G,H,p,q) # Equilibrium radius (where F = 0)
r_c = 1.3*r_0  
dx = r_0
rMin = np.float64(dx *0.1)  
r_skin = np.float64(dx *0.5)
kappa = R_in / R_out
gap = R_out - R_in # Gap de 1 mm         
# Velocidades 
vp = 1e-7         
omega = 0.0628  
# Tiempos
h = 1e-4
totalSteps = 1000         
frameIterSep = 100     
total_simulation = totalSteps*h  
# Colisión
damp = 0.75
threshold = 0.01 
# Altura
L = 1.0           
n_layers = int(L // dx)  
d = dx           
particle_size = 20   
dz = L / (n_layers-1) 
print(f"Número de filas verticales: {n_layers}")
print(f"Distancia entre capas: {dz} cm")
print(f"Distancia de interacción: {r_c} cm")
print(f"\kappa: {kappa}")
print(f"Distancia entre particulas x e y : {dx} cm")

El radio de equilibrio exacto es: 0.05 cm
Número de filas verticales: 19
Distancia entre capas: 0.05555555555555555 cm
Distancia de interacción: 0.065 cm
\kappa: 0.9261992619926198
Distancia entre particulas x e y : 0.05 cm


## Funciones

In [4]:
# ============================================================
# FUNCIONES GENERADORAS DE PARTICULAS EN 3D
# ============================================================

def gen_pp_3d(radius, height, n_layers, spacing):
    """
    Generate particle positions in a 3D cylindrical arrangement.
    
    Parameters:
    - radius: Cylinder radius
    - height: Total height
    - n_layers: Number of layers
    - spacing: Particle spacing
    
    Returns:
    - positions: Array of particle positions (N, 3)
    - colors: List of colors for each particle based on row parity
    - num_particles: Total number of particles
    """
    dz = height / (n_layers - 1)
    
    # Función que genera los puntos para una capa dada (2D)
    def generate_layer(r, sp, z, offset=False):
        points = []
        colors = []
        dy = r_0  # Vertical spacing between rows
        y_vals = np.arange(-r, r + dy, dy)
        for j, y in enumerate(y_vals):
            shift = sp / 2 if (j % 2 == 1) else 0
            if offset:
                shift += sp / 2
            x_vals = np.arange(-r, r + sp, sp)
            for x in x_vals:
                xx = x + shift
                if R_in**2 <= xx**2 + y**2 <= R_out**2:
                    points.append([xx, y, z])
                    colors.append('red' if j % 2 == 0 else 'blue')  # Even rows red, odd blue
        return np.array(points), colors
    
    all_points = []
    all_colors = []
    for k in range(n_layers):
        z = k * dz
        offset = (k % 2 == 1)  # variable lógica, si se cumple, se vuelve true, si no, false
        layer_points, layer_colors = generate_layer(radius, spacing, z, offset)
        all_points.append(layer_points)
        all_colors.extend(layer_colors)
    
    positions = np.vstack(all_points)
    N = len(positions)
    
    return positions, all_colors, N

In [5]:
@njit
def build_verlet_list_mod_numba(positions, r_cutoff, skin_thickness):
    """Construye la lista de Verlet con dos recorridos compilados."""
    n = positions.shape[0]
    r_list = r_cutoff + skin_thickness
    r_list_sq = r_list * r_list

    # Primer recorrido: cuenta vecinos por partícula para reservar memoria.
    counts = np.zeros(n, dtype=np.int64)

    for i in range(n - 1): # i va hasta n-1 porque la ultima partícula no tiene vecinos por delante
        for j in range(i + 1, n):
            dx = positions[j, 0] - positions[i, 0]
            dy = positions[j, 1] - positions[i, 1]
            dz = positions[j, 2] - positions[i, 2]
            dist_sq = dx * dx + dy * dy + dz * dz

            if dist_sq < r_list_sq:
                counts[i] += 1

    # Mantiene el formato de `point` de la función original.
    # Point tiene N+1 porque indices porque en python 
    point = np.zeros(n + 1, dtype=np.int64)
    for i in range(n - 1):
        point[i + 1] = point[i] + counts[i]

    # n_pairs es el número total de pares de vecinos encontrados, se resta 1 porque el último índice de point es 0
    n_pairs = point[n - 1] if n > 0 else 0
    # creamos un arreglo vacío para almacenar los vecinos de Verlet (ahorramos espacio al saber los pares ya identificados)
    verlet_neighbors = np.empty(n_pairs, dtype=np.int64)

    # Cursor es el arreglo donde se almacena el índice del siguiente vecino para cada partícula
    cursor = point[:n].copy()

    for i in range(n - 1):
        for j in range(i + 1, n):
            dx = positions[j, 0] - positions[i, 0]
            dy = positions[j, 1] - positions[i, 1]
            dz = positions[j, 2] - positions[i, 2]
            dist_sq = dx * dx + dy * dy + dz * dz

            if dist_sq < r_list_sq:
                verlet_neighbors[cursor[i]] = j
                cursor[i] += 1

    return verlet_neighbors, point

In [6]:
# Listas de Verlet (Tildesley)
def build_verlet_list_mod(positions, r_cutoff, skin_thickness):
    """
    Construye la lista de vecinos de Verlet (O(N^2)).
    r_list = r_cutoff + skin_thickness (corresponde a r_l en el libro).
    """
    N = len(positions)
    r_list_sq = (r_cutoff + skin_thickness)**2
    verlet_neighbors = []
    point = np.zeros(N + 1, dtype=int)
    
    for i in range(N - 1):
        diff = positions[i+1:] - positions[i]
        dist_sq = np.sum(diff**2, axis=1)
        
        # Identificar índices de vecinos dentro del radio extendido
        neighbors = np.where(dist_sq < r_list_sq)[0] + (i + 1)

        # verlet_neighbors es la lista de todos los vecinos de todas las partículas [i]
        verlet_neighbors.extend(neighbors) #Extend añade los vecinos al final
        # El puntero apunta al inicio de los vecinos de la siguiente partícula [3]
        point[i+1] = point[i] + len(neighbors)
    # Retorna un array de enteros con los vecinos y un array de punteros que indica dónde comienzan los vecinos de cada partícula
    return np.array(verlet_neighbors, dtype=int), point
        

In [7]:
@njit
def _lj_force_scalar_numba(
    body1, body2, r_cutoff, r_min, exponent_p, exponent_q, G, H
):
    dx = body2[0] - body1[0]
    dy = body2[1] - body1[1]
    dz = body2[2] - body1[2]

    distance_sq = dx * dx + dy * dy + dz * dz
    cutoff_sq = r_cutoff * r_cutoff

    if distance_sq < 1e-12 or distance_sq > cutoff_sq:
        return 0.0, 0.0, 0.0

    distance = np.sqrt(distance_sq)

    if distance < r_min:
        distance = r_min

    inv_distance = 1.0 / distance
    inv_r_power = 1.0
    inv_r_p = 1.0 if exponent_p == 0 else 0.0

    for exponent in range(1, exponent_q + 1):
        inv_r_power *= inv_distance
        if exponent == exponent_p:
            inv_r_p = inv_r_power

    inv_r_q = inv_r_power
    force_scale = -G * inv_r_p + H * inv_r_q

    return dx * force_scale, dy * force_scale, dz * force_scale

In [8]:
def lj_force_scalar(body1, body2):
    x1, y1, z1 = body1[0], body1[1], body1[2]
    x2, y2, z2 = body2[0], body2[1], body2[2]

    dx = x2 - x1
    dy = y2 - y1
    dz = z2 - z1  
    distance_sq = dx * dx + dy * dy + dz * dz

    if distance_sq < 1e-12 or distance_sq > r_c * r_c:
        return 0.0, 0.0, 0.0

    distance = np.sqrt(distance_sq)

    if distance < rMin:
        distance = rMin

    inv_distance = 1.0 / distance
    inv_r_power = 1.0
    inv_r_p = 1.0 if p == 0 else 0.0

    for exponent in range(1, q + 1):
        inv_r_power *= inv_distance
        if exponent == p:
            inv_r_p = inv_r_power

    inv_r_q = inv_r_power
    force_scale = -G * inv_r_p + H * inv_r_q

    return dx * force_scale, dy * force_scale, dz * force_scale

In [9]:
def compute_net_forces_verlet_mod(positions, verlet_neighbors, point, r_cutoff):
    N = len(positions)
    forces = np.zeros_like(positions)
    r_cut_sq = r_cutoff**2
    
    for i in range(N - 1):
        # Extraer vecinos de la partícula i desde el arreglo list [6]
        for idx in range(point[i], point[i+1]):
            j = verlet_neighbors[idx]
            
            rij = positions[i] - positions[j]
            dist_sq = np.sum(rij**2)
            # Checa solo los vecino dentro del radio de corte
            if dist_sq < r_cut_sq:
                # Aquí se aplica la ley de fuerza (ej. Greenspan o LJ) [7]
                f_vec = lj_force_scalar(positions[i], positions[j]) 
                forces[i] += f_vec
                forces[j] -= f_vec # Aplicación de la 3ra Ley de Newton [5]
                
    return forces

In [10]:
@njit
def _compute_net_forces_numba(
    positions,
    verlet_neighbors,
    point,
    r_cutoff,
    r_min,
    exponent_p,
    exponent_q,
    G,
    H,
):
    n = positions.shape[0]
    forces = np.zeros_like(positions)
    cutoff_sq = r_cutoff * r_cutoff

    for i in range(n - 1):
        for idx in range(point[i], point[i + 1]):
            j = verlet_neighbors[idx]

            dx = positions[i, 0] - positions[j, 0]
            dy = positions[i, 1] - positions[j, 1]
            dz = positions[i, 2] - positions[j, 2]
            dist_sq = dx * dx + dy * dy + dz * dz

            if dist_sq < cutoff_sq:
                fx, fy, fz = _lj_force_scalar_numba(
                    positions[i],
                    positions[j],
                    r_cutoff,
                    r_min,
                    exponent_p,
                    exponent_q,
                    G,
                    H,
                )

                forces[i, 0] += fx
                forces[i, 1] += fy
                forces[i, 2] += fz

                forces[j, 0] -= fx
                forces[j, 1] -= fy
                forces[j, 2] -= fz

    return forces

In [11]:
def compute_net_forces_verlet_mod_numba(
    positions, verlet_neighbors, point, r_cutoff
):
    # La función de Python pasa los parámetros del modelo al kernel compilado.
    return _compute_net_forces_numba(
        positions,
        verlet_neighbors,
        point,
        r_cutoff,
        rMin,
        p,
        q,
        G,
        H,
    )

In [12]:
def should_rebuild_list(current_pos, last_build_pos, skin_thickness):
    """
    Criterio de actualización de Tildesley (2017) 
    """
    # Magnitud de desplazamiento individual para cada partícula
    displacements = np.sqrt(np.sum((current_pos - last_build_pos)**2, axis=1))

    # Obtener los dos mayores desplazamientos (O(N))
    top_two = np.partition(displacements, -2)[-2:]
    
    # Condición: suma de los dos mayores > espesor de la piel [4]
    return np.sum(top_two) > skin_thickness

In [13]:
def apply_boundary_velocity_filter_3D(positions, velocities, threshold=0.01 * dx):

    # Aunque sea 3D, no necesitamos la componente z para saber que tan cerca esta de la pared en el plano xy
    r2 = positions[:, 0]**2 + positions[:, 1]**2
    r = np.sqrt(r2)
    z = positions[:, 2]
    theta = np.arctan2(positions[:, 1], positions[:, 0])
    boundary_mask = (
        (np.abs(r - R_in) < threshold)
        & (z >= 0) 
        & (z <= L)
    )
    if np.any(boundary_mask):
        velocities[boundary_mask, 0] = -omega * r[boundary_mask] * np.sin(theta[boundary_mask])
        velocities[boundary_mask, 1] = omega * r[boundary_mask] * np.cos(theta[boundary_mask])
        velocities[boundary_mask, 2] = 0.0
    
    return velocities, boundary_mask 

In [ ]:
def keepInside(position, velocity, newPosition, newVelocity):
    # copia entradas
    newPosition[:] = position
    newVelocity[:] = velocity

    # Distancia
    r_xy = newPosition[:, 0]**2 + newPosition[:, 1]**2 

    # ----- Máscaras ----
    outer_mask = r_xy > R_out**2
    inner_mask = r_xy <= R_in**2

    # Aqui plantear la condición de extremo superior e inferior
    # Máscaras para las tapas del cilindro
    bottom_mask = newPosition[:, 2] < 0.0
    top_mask = newPosition[:, 2] > L

    eps = 1e-12 # Tolerancia para evitar división entre cero si r -&gt; 0
    
    # --- Pared inferior (z = 0) --- if np.any(bottom\_mask): # Reflejo de posición: la partícula rebota hacia adentro newPosition[bottom\_mask, 2] = -newPosition[bottom\_mask, 2] # Choque inelástico: invierte vz y amortigua vx, vy newVelocity[bottom\_mask, 2] = -damp \* newVelocity[bottom\_mask, 2] newVelocity[bottom\_mask, 0] \*= damp newVelocity[bottom\_mask, 1] \*= damp # --- Pared superior (z = L) --- if np.any(top\_mask): # Reflejo de posición: 2L - z newPosition[top\_mask, 2] = 2.0 \* L - newPosition[top\_mask, 2] # Choque inelástico: invierte vz y amortigua vx, vy newVelocity[top\_mask, 2] = -damp \* newVelocity[top\_mask, 2] newVelocity[top\_mask, 0] \*= damp newVelocity[top\_mask, 1] \*= damp
        
    if np.any(outer_mask):
        r_outer = np.sqrt(r_xy[outer_mask])
        r_outer = np.maximum(r_outer, eps)  # Evitar división por cero

        normal_x = newPosition[outer_mask, 0] / r_outer
        normal_y = newPosition[outer_mask, 1] / r_outer
        tangential_x = -normal_y
        tangential_y = normal_x

        # 1. Corrección de posición
        new_r = 2.0 * R_out - r_outer
        newPosition[outer_mask, 0] = new_r * normal_x
        newPosition[outer_mask, 1] = new_r * normal_y
        
        # 2. Corrección de velocidad (reflexión con amortiguamiento)
        vn_mag = newVelocity[outer_mask, 0] * normal_x + newVelocity[outer_mask, 1] * normal_y
        vt_mag = newVelocity[outer_mask, 0] * tangential_x + newVelocity[outer_mask, 1] * tangential_y

        new_vn = -damp * vn_mag
        new_vt = damp * vt_mag # No hay inyección de momento en la pared externa

        newVelocity[outer_mask, 0] = new_vn * normal_x + new_vt * tangential_x
        newVelocity[outer_mask, 1] = new_vn * normal_y + new_vt * tangential_y

    # choque/reflexión interna con pared giratoria
    if np.any(inner_mask):
        # Radio penetrado actual O(R_i)
        r_inner = np.sqrt(r_xy[inner_mask])
        r_inner = np.maximum(r_inner, eps)  # Evitar división por cero
        # Vectores unitarios normal y tangencial
        normal_x = newPosition[inner_mask, 0] / r_inner
        normal_y = newPosition[inner_mask, 1] / r_inner
        tangential_x = -normal_y
        tangential_y = normal_x

        # 2. CORRECCIÓN DE POSICIONES (Reflejo simétrico de Greenspan/Mahmoudi)
        r_new = 2.0 * R_in - r_inner
        newPosition[inner_mask, 0] = normal_x * r_new
        newPosition[inner_mask, 1] = normal_y * r_new

        # 3. CORRECCIÓN DE VELOCIDADES (Inyección de momento)
        v_wall_mag = omega * R_in
        
        vn_mag = newVelocity[inner_mask, 0] * normal_x + newVelocity[inner_mask, 1] * normal_y
        vt_mag = newVelocity[inner_mask, 0] * tangential_x + newVelocity[inner_mask, 1] * tangential_y
        
        new_vn = -damp * vn_mag
        new_vt = damp * vt_mag + (1.0 - damp) * v_wall_mag

        newVelocity[inner_mask, 0] = new_vn * normal_x + new_vt * tangential_x
        newVelocity[inner_mask, 1] = new_vn * normal_y + new_vt * tangential_y

    return newPosition, newVelocity

In [15]:
def getInitialVelocities3D(num_particles, positions, threshold=0.01 * dx):

    # Uniformly distributed directions on a sphere
    theta = np.random.uniform(0, 2 * np.pi, num_particles)
    # Uniformly distributed polar angles using the inverse transform method
    phi = np.arccos(
        1 - 2 * np.random.random(num_particles)
    )

    velocities = np.zeros((num_particles, 3), dtype=np.float64)

    velocities[:, 0] = vp * np.sin(phi) * np.cos(theta)
    velocities[:, 1] = vp * np.sin(phi) * np.sin(theta)
    velocities[:, 2] = vp * np.cos(phi)

    # Apply the moving-wall boundary treatment
    velocities, boundary_mask = apply_boundary_velocity_filter_3D(
        positions,
        velocities,
        threshold
    )

    print(
        f"Partículas ajustadas cerca de la pared interna: "
        f"{np.sum(boundary_mask)} de {num_particles}"
    )

    return velocities

## Simulación

In [16]:
from math import e
import time

#from cycler import K

# El if __name__ == '__main__': asegura que el código dentro de este bloque solo se ejecute cuando el script se ejecute directamente, y no cuando se importe como módulo en otro script. Esto es útil para evitar la ejecución de código no deseado durante la importación.
if __name__ == '__main__':
    timing = {
        "construccion_lista_vecinos": 0.0,
        "calculo_fuerzas": 0.0,
        "integrador": 0.0,
        "keepInside": 0.0,
        "verificacion_reconstruccion": 0.0,
        "guardado": 0.0,
    }
    calls = {name: 0 for name in timing}

    # Empieza a contar el tiempo de la simulación
    t_start = time.perf_counter()

    # Parámetros de tiempo 
    hstep = h 
    startIndex = 0

    # Genera las posiciones y velocidades iniciales
    bodies_dataX, colors, N = gen_pp_3d(R_out, L, n_layers, d)
    numBodies = N
    bodies_dataV = getInitialVelocities3D(numBodies, bodies_dataX, threshold=0.01 * dx)

    # Información inicial de la simulación
    print("=" * 70)
    print("INICIANDO SIMULACIÓN")
    print("=" * 70)
    print(f"Forma de la información de X: {bodies_dataX.shape}")
    print(f"Forma de la información de V: {bodies_dataV.shape}")

    # Construye la lista de vecinos de Verlet
    neighbor_pairs, point = build_verlet_list_mod_numba(bodies_dataX,r_c, r_skin)
    # Construimos el arreglo de last_built_positions, será una copia de las posiciones
    last_built_positions = bodies_dataX.copy()

    # Obtén las aceleraciones en función de las posiciones iniciales y la lista de vecinos)
    bodies_dataA = compute_net_forces_verlet_mod_numba(bodies_dataX, neighbor_pairs, point, r_c)
    index = int(0)

    # Empezamos a guardar los archivos binarios para guardar las posiciones y velocidades de las partículas rápidas
    with open("C:/Users/ivan_/Desktop/Tesis/Python_scripts/Jupyter/3d_rheometer/Positions.bin", 'wb') as sp:
        with open("C:/Users/ivan_/Desktop/Tesis/Python_scripts/Jupyter/3d_rheometer/Velocities.bin", 'wb') as sv:
            with open("C:/Users/ivan_/Desktop/Tesis/Python_scripts/Jupyter/3d_rheometer/serialTime.bin", 'wb') as st:
                # Definimos la variable de frames para guardar cada frame con separación de frameIterSep pasos    
                frames = 0
                for s in range(startIndex, totalSteps):
                    
                    """
                    # MONITOREO DE LA SIMULACION
                    print(f"\nTiempo actual de la simulación: {np.float64(s*hstep)}")
                    aceleraciones = np.sqrt(bodies_dataA[:,0]**2 + bodies_dataA[:,1]**2)
                    aceleracion_max = np.max(aceleraciones)
                    print(f"Aceleración máxima para tiempo  {np.float64(s*hstep)} es:  {aceleracion_max}")
                    aceleracion_min = np.min(aceleraciones)
                    print(f"Aceleración mínima para tiempo  {np.float64(s*hstep)} es:  {aceleracion_min}")                        
                    aceleracion_media = np.mean(aceleraciones)
                    print(f"Aceleración media para tiempo  {np.float64(s*hstep)} es:  {aceleracion_media}")
                    """

                    """ ESTO OCUPA TIEMPO DE SIMULACIÓN TAMBIÉN
                    # Checamos velocidades promedio 
                    print(f"\nTiempo actual de la simulación: {np.float64(s*hstep)}")
                    velocidades = np.sqrt(bodies_dataV[:,0]**2 + bodies_dataV[:,1]**2+bodies_dataV[:,2]**2)
                    velocidad_max = np.max(velocidades)
                    print(f"Velocidad máxima para tiempo  {np.float64(s*hstep)} es:  {velocidad_max}")
                    velocidad_min = np.min(velocidades)
                    print(f"Velocidad mínima para tiempo  {np.float64(s*hstep)} es:  {velocidad_min}")                        
                    velocidad_media = np.mean(velocidades)
                    print(f"Velocidad media para tiempo  {np.float64(s*hstep)} es:  {velocidad_media}")
                    """

                    # ---------------- INTEGRADOR ------------------------------
                    # Aplicamos el metodo de Leapfrog (Verlet)
                    t0 = time.perf_counter()
                    bodies_dataV = bodies_dataV + 0.5*hstep*bodies_dataA
                    bodies_dataX = bodies_dataX + hstep*bodies_dataV
                    timing["integrador"] += time.perf_counter() - t0
                    calls["integrador"] += 1

                    """
                    # ---------------- ENERGIA ------------------------------
                    # 2. DETECCIÓN DE FRONTERAS Y CÁLCULO DE ENERGÍA (Antes de aplicar keepInside)
                    radii = np.sqrt(bodies_dataX[:, 0]**2 + bodies_dataX[:, 1]**2)
                
                    # First check wheter particles are inside vertical boundaries (z=0 and z=L) and apply boundary conditions
                    z = bodies_dataX[:, 2]
                    bottom_mask = z < 0.0
                    top_mask = z > L
                    if np.any(bottom_mask):
                    # A. COLISIONES CON EL CILINDRO EXTERNO (Copa)
                    idx_out = np.where(radii > R_out)
                    total_E_dissipated = 0.0
                    total_E_injected = 0.0
                    if len(idx_out) > 0:
                        # Velocidad antes del choque
                        v_before = bodies_dataV[idx_out]
                        v_mag2 = np.sum(v_before**2, axis=1)
                        # Energía cinética disipada en este paso (como valor positivo)
                        # E_dis = 0.5 * m * v^2 * (1 - alpha^2)
                        E_lost = 0.5 * m * v_mag2 * (1.0 - damp**2)
                        total_E_dissipated += np.sum(E_lost)
                    
                    # B. COLISIONES CON EL CILINDRO INTERNO (Vástago)
                    idx_in = np.where(radii <= R_in)
                    if len(idx_in) > 0:
                        # Datos antes del choque
                        x_before = bodies_dataX[idx_in]
                        v_before = bodies_dataV[idx_in]
                        r_mag = radii[idx_in][:, np.newaxis]
                        
                        # Componentes normal y tangencial
                        n = x_before / r_mag
                        t = np.column_stack((-n[:, 1], n[:, 0]))
                        
                        vn = np.sum(v_before * n, axis=1)
                        vt = np.sum(v_before * t, axis=1)
                        
                        v_wall = omega * R_in
                        
                        # Calcular velocidad resultante teórica tras el operador B(x,v)
                        vn_new = -damp * vn
                        vt_new = damp * vt + (1.0 - damp) * v_wall
                        
                        # Energía cinética antes y después
                        K_before = 0.5 * m * np.sum(v_before**2, axis=1)
                        K_after = 0.5 * m * (vn_new**2 + vt_new**2)
                        
                        delta_K = K_after - K_before
                        
                        # Separar en inyección neta o amortiguamiento del cilindro interno
                        # (Si delta_K es positivo, es energía neta inyectada al sistema)
                        total_E_injected += np.sum(np.clip(delta_K, a_min=0, a_max=None))
                        total_E_dissipated += np.sum(np.clip(-delta_K, a_min=0, a_max=None))
                    """
                    # ------------------- KEEP PARTICLES INSIDE RHEOMETER
                    t0 = time.perf_counter()
                    bodies_dataX, bodies_dataV = keepInside(bodies_dataX, bodies_dataV, 
                                                np.zeros_like(bodies_dataX), 
                                                np.zeros_like(bodies_dataV))
                    timing["keepInside"] += time.perf_counter() - t0
                    calls["keepInside"] += 1

                    # ------------------ REBUILD VERLET LIST ------------------------------
                    t0 = time.perf_counter()
                    # Calculamos el desplazamiento cuadrático de cada partícula desde la última vez que se construyó la lista de vecinos
                    top_two_sq = should_rebuild_list(bodies_dataX, last_built_positions, r_skin)
                    timing["verificacion_reconstruccion"] += time.perf_counter() - t0
                    calls["verificacion_reconstruccion"] += 1
 
                    if top_two_sq:
                        t0 = time.perf_counter()
                        neighbor_pairs, point = build_verlet_list_mod_numba(bodies_dataX, r_c, r_skin)
                        last_built_positions = bodies_dataX.copy()
                        timing["construccion_lista_vecinos"] += time.perf_counter() - t0
                        calls["construccion_lista_vecinos"] += 1

                    # ------------------ CALCULO DE FUERZAS ------------------------------
                    # Para las partículas reflejadas, calculamos su fuerza, aceleración y nuevas velocidades
                    t0 = time.perf_counter()
                    bodies_dataA = compute_net_forces_verlet_mod_numba(bodies_dataX, neighbor_pairs, point, r_c)
                    timing["calculo_fuerzas"] += time.perf_counter() - t0
                    calls["calculo_fuerzas"] += 1

                    # Segunda mitad del integrador de Leapfrog (Verlet)
                    t0 = time.perf_counter()
                    bodies_dataV = bodies_dataV + 0.5*hstep*bodies_dataA
                    timing["integrador"] += time.perf_counter() - t0
                    
                    # Guardamos el tiempo de la simulación en un archivo binario
                    if (s + 1) % frameIterSep == 0:
                    # Guardar estado y tiempo del paso completado
                        t0 = time.perf_counter()
                        sp.write(bodies_dataX.tobytes())
                        sv.write(bodies_dataV.tobytes())
                        st.write(np.float64((s + 1) * hstep).tobytes())
                        timing["guardado"] += time.perf_counter() - t0
                        calls["guardado"] += 1
                frames += 1

    # Comportamiento de la simulación
    print(f"Número de partículas: {N}")
    t_end = time.perf_counter()
    elapsed_time = t_end - t_start
    print(f"\n⏱️  Tiempo en correr la simulación: {elapsed_time:.2f} segundos || {elapsed_time/60:.2f} minutos || {elapsed_time/3600:.2f} horas")                
    actual_steps = max(1,totalSteps - startIndex)
    frames_saved = max(1,(actual_steps // frameIterSep)) 
    avg_per_step = elapsed_time / actual_steps
    avg_per_frame = elapsed_time / frames_saved

    print(f"\n" + "="*70)
    print(f"Tiempo total de ejecución (wall-clock): {elapsed_time:.3f} s")
    print(f"Tiempo promedio por paso: {avg_per_step*1000:.6f} ms")
    print(f"Tiempo promedio por frame guardado: {avg_per_frame:.3f} s")

    print("\nTiempo acumulado por operación:")
    for name, total_seconds in timing.items():
        count = calls[name]
        average_ms = (
            total_seconds * 1000 / count
            if count > 0
            else 0.0
        )
        print(
            f"{name}: total={total_seconds:.3f} s, "
            f"llamadas={count}, promedio={average_ms:.3f} ms"
        )

    # Guardamos los datos de la simulación en un archivo txt
    with open('C:/Users/ivan_/Desktop/Tesis/Python_scripts/Jupyter/3d_rheometer/sim_data.txt', 'w') as f:
        f.write(f"totalSteps: {totalSteps}\n")
        f.write(f"sim_time: {totalSteps*h:.5f} segundos\n")
        f.write(f"frameIterSep: {frameIterSep} pasos\n")
        f.write(f"total_frames: {totalSteps//frameIterSep}\n")
        f.write(f"H: {H}\n")
        f.write(f"G: {G}\n")
        f.write(f"p: {p}\n")
        f.write(f"q: {q}\n")
        f.write(f"delta_x: {dx} cm\n")
        f.write(f"vp: {vp} cm/s\n")
        f.write(f"R_in: {R_in} cm\n")
        f.write(f"R_out: {R_out} cm\n")
        f.write(f"damp: {damp}\n")
        f.write(f"r_c: {r_c} cm\n")
        f.write(f"rMin: {rMin} cm\n")
        f.write(f"N: {N}\n")
        f.write(f"texec_time: {elapsed_time/60:.3f} min\n")
        f.write(f"m: {m}\n")
        f.write(f"r_0: {r_0} cm\n")
        f.write(f"dx: {dx} cm\n")
        f.write(f"r_skin: {r_skin} cm\n")
        f.write(f"kappa: {kappa}\n")
        f.write(f"gap: {gap} cm\n")
        f.write(f"omega: {omega} rad/s\n")
        f.write(f"h: {h} s\n")
        f.write(f"total_simulation: {total_simulation} s\n")
        f.write(f"threshold: {threshold}\n")
        f.write(f"L: {L} cm\n")
        f.write(f"n_layers: {n_layers}\n")
        f.write(f"d: {d} cm\n")
        f.write(f"particle_size: {particle_size}\n")
        f.write(f"dz: {dz} cm\n")
        for name, total_seconds in timing.items():
                count = calls[name]
                average_ms = (
                    total_seconds * 1000 / count
                    if count > 0
                    else 0.0
                )
                f.write(f"{name}: total={total_seconds:.3f} s")
                f.write(f"llamadas={count}, promedio={average_ms:.3f} ms\n")
            


print(f"\n" + "="*70)
print(f"✅ SIMULACIÓN COMPLETADA")
print(f"="*70)

Partículas ajustadas cerca de la pared interna: 122 de 6293
INICIANDO SIMULACIÓN
Forma de la información de X: (6293, 3)
Forma de la información de V: (6293, 3)
Número de partículas: 6293

⏱️  Tiempo en correr la simulación: 54.81 segundos || 0.91 minutos || 0.02 horas

Tiempo total de ejecución (wall-clock): 54.811 s
Tiempo promedio por paso: 54.810961 ms
Tiempo promedio por frame guardado: 5.481 s

Tiempo acumulado por operación:
construccion_lista_vecinos: total=47.956 s, llamadas=1000, promedio=47.956 ms
calculo_fuerzas: total=0.866 s, llamadas=1000, promedio=0.866 ms
integrador: total=0.078 s, llamadas=1000, promedio=0.078 ms
keepInside: total=0.493 s, llamadas=1000, promedio=0.493 ms
verificacion_reconstruccion: total=0.175 s, llamadas=1000, promedio=0.175 ms
guardado: total=0.002 s, llamadas=10, promedio=0.222 ms

✅ SIMULACIÓN COMPLETADA
